# Dragon Real Estates — End-to-End Machine Learning Project

This notebook is an original reconstruction of the workflow covered in the referenced tutorial video.

It covers:
- Loading and inspecting the housing dataset
- Train/test splitting
- Stratified sampling
- Correlation analysis and visualization
- Missing-value handling
- Preprocessing pipelines
- Linear Regression
- Decision Tree Regression
- Random Forest Regression
- Cross-validation
- Final test evaluation
- Saving and loading the trained model

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pandas.plotting import scatter_matrix

from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
    cross_val_score
)

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

import joblib

## 2. Load the housing dataset

In [ ]:
housing = pd.read_csv("data.csv")

In [ ]:
housing.head()

In [ ]:
housing.columns

## 3. Inspect the dataset

In [ ]:
housing.info()

In [ ]:
housing.describe()

In [ ]:
housing.shape

In [ ]:
housing.isnull().sum()

## 4. Examine the CHAS attribute

In [ ]:
housing["CHAS"].value_counts()

In [ ]:
housing["CHAS"].value_counts(normalize=True)

## 5. Plot histograms

In [ ]:
housing.hist(bins=50, figsize=(20, 15))
plt.tight_layout()
plt.show()

## 6. Ordinary train/test split

In [ ]:
train_set, test_set = train_test_split(
    housing,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(train_set))
print("Testing rows :", len(test_set))

## 7. Manual train/test split concept

In [ ]:
def split_train_test(data, test_ratio, random_state=None):
    rng = np.random.default_rng(random_state)

    shuffled_indices = rng.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)

    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]

    return data.iloc[train_indices], data.iloc[test_indices]

In [ ]:
manual_train_set, manual_test_set = split_train_test(
    housing,
    test_ratio=0.2,
    random_state=42
)

print(len(manual_train_set))
print(len(manual_test_set))

## 8. Stratified Shuffle Split

In [ ]:
split = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_index, test_index in split.split(
    housing,
    housing["CHAS"]
):
    strat_train_set = housing.iloc[train_index].copy()
    strat_test_set = housing.iloc[test_index].copy()

In [ ]:
strat_train_set["CHAS"].value_counts()

In [ ]:
strat_test_set["CHAS"].value_counts()

In [ ]:
strat_train_set["CHAS"].value_counts(normalize=True)

In [ ]:
strat_test_set["CHAS"].value_counts(normalize=True)

## 9. Create a working copy

In [ ]:
housing_explore = strat_train_set.copy()

## 10. Correlation analysis

In [ ]:
corr_matrix = housing_explore.corr(numeric_only=True)

In [ ]:
corr_matrix["MEDV"].sort_values(ascending=False)

## 11. Scatter-matrix visualization

In [ ]:
attributes = [
    "MEDV",
    "RM",
    "ZN",
    "LSTAT"
]

scatter_matrix(
    housing_explore[attributes],
    figsize=(12, 8)
)

plt.show()

## 12. Rooms versus price

In [ ]:
housing_explore.plot(
    kind="scatter",
    x="RM",
    y="MEDV",
    alpha=0.8
)

plt.title("Average Rooms vs House Price")
plt.show()

## 13. LSTAT versus price

In [ ]:
housing_explore.plot(
    kind="scatter",
    x="LSTAT",
    y="MEDV",
    alpha=0.8
)

plt.title("LSTAT vs House Price")
plt.show()

## 14. Experiment with attribute combinations

In [ ]:
housing_explore["TAXRM"] = (
    housing_explore["TAX"] /
    housing_explore["RM"]
)

In [ ]:
corr_matrix = housing_explore.corr(numeric_only=True)

corr_matrix["MEDV"].sort_values(ascending=False)

In [ ]:
housing_explore.plot(
    kind="scatter",
    x="TAXRM",
    y="MEDV",
    alpha=0.8
)

plt.show()

## 15. Separate features and labels

In [ ]:
housing_features = strat_train_set.drop(
    "MEDV",
    axis=1
)

housing_labels = strat_train_set[
    "MEDV"
].copy()

In [ ]:
print(housing_features.shape)
print(housing_labels.shape)

## 16. Missing-value demonstration

In [ ]:
housing_features.isnull().sum()

In [ ]:
# Option 1:
# housing_features.dropna(subset=["RM"])

# Option 2:
# housing_features.drop("RM", axis=1)

# Option 3:
# median = housing_features["RM"].median()
# housing_features["RM"] = housing_features["RM"].fillna(median)

## 17. SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy="median")

In [ ]:
imputer.fit(housing_features)

In [ ]:
imputer.statistics_

In [ ]:
housing_features.median(numeric_only=True).values

In [ ]:
X = imputer.transform(housing_features)

In [ ]:
housing_tr = pd.DataFrame(
    X,
    columns=housing_features.columns,
    index=housing_features.index
)

In [ ]:
housing_tr.head()

## 18. Build the preprocessing pipeline

In [ ]:
my_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "std_scaler",
        StandardScaler()
    )
])

In [ ]:
housing_prepared = my_pipeline.fit_transform(
    housing_features
)

In [ ]:
housing_prepared.shape

In [ ]:
housing_prepared[:5]

## 19. Linear Regression model

In [ ]:
lin_reg = LinearRegression()

In [ ]:
lin_reg.fit(
    housing_prepared,
    housing_labels
)

## 20. Try predictions on a few examples

In [ ]:
some_data = housing_features.iloc[:5]
some_labels = housing_labels.iloc[:5]

some_data_prepared = my_pipeline.transform(
    some_data
)

In [ ]:
lin_predictions = lin_reg.predict(
    some_data_prepared
)

lin_predictions

In [ ]:
list(some_labels)

In [ ]:
comparison = pd.DataFrame({
    "Actual": some_labels.values,
    "Predicted": lin_predictions
})

comparison

## 21. Evaluate Linear Regression

In [ ]:
housing_predictions = lin_reg.predict(
    housing_prepared
)

lin_mse = mean_squared_error(
    housing_labels,
    housing_predictions
)

lin_rmse = np.sqrt(lin_mse)

lin_rmse

In [ ]:
lin_r2 = r2_score(
    housing_labels,
    housing_predictions
)

lin_r2

## 22. Decision Tree Regressor

In [ ]:
tree_reg = DecisionTreeRegressor(
    random_state=42
)

In [ ]:
tree_reg.fit(
    housing_prepared,
    housing_labels
)

In [ ]:
housing_predictions = tree_reg.predict(
    housing_prepared
)

In [ ]:
tree_mse = mean_squared_error(
    housing_labels,
    housing_predictions
)

tree_rmse = np.sqrt(tree_mse)

tree_rmse

## 23. Cross-validation

In [ ]:
tree_scores = cross_val_score(
    tree_reg,
    housing_prepared,
    housing_labels,
    scoring="neg_mean_squared_error",
    cv=10
)

tree_rmse_scores = np.sqrt(
    -tree_scores
)

In [ ]:
def display_scores(scores):
    print("Scores:")
    print(scores)

    print("\nMean:")
    print(scores.mean())

    print("\nStandard deviation:")
    print(scores.std())

In [ ]:
display_scores(tree_rmse_scores)

## 24. Cross-validation for Linear Regression

In [ ]:
lin_scores = cross_val_score(
    lin_reg,
    housing_prepared,
    housing_labels,
    scoring="neg_mean_squared_error",
    cv=10
)

lin_rmse_scores = np.sqrt(
    -lin_scores
)

In [ ]:
display_scores(lin_rmse_scores)

## 25. Random Forest Regressor

In [ ]:
forest_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [ ]:
forest_reg.fit(
    housing_prepared,
    housing_labels
)

In [ ]:
housing_predictions = forest_reg.predict(
    housing_prepared
)

forest_mse = mean_squared_error(
    housing_labels,
    housing_predictions
)

forest_rmse = np.sqrt(
    forest_mse
)

forest_rmse

## 26. Cross-validation for Random Forest

In [ ]:
forest_scores = cross_val_score(
    forest_reg,
    housing_prepared,
    housing_labels,
    scoring="neg_mean_squared_error",
    cv=10,
    n_jobs=-1
)

forest_rmse_scores = np.sqrt(
    -forest_scores
)

In [ ]:
display_scores(
    forest_rmse_scores
)

## 27. Compare the models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "CV Mean RMSE": [
        lin_rmse_scores.mean(),
        tree_rmse_scores.mean(),
        forest_rmse_scores.mean()
    ],
    "CV Std": [
        lin_rmse_scores.std(),
        tree_rmse_scores.std(),
        forest_rmse_scores.std()
    ]
})

results.sort_values("CV Mean RMSE")

## 28. Prepare the test set

In [ ]:
X_test = strat_test_set.drop(
    "MEDV",
    axis=1
)

y_test = strat_test_set[
    "MEDV"
].copy()

In [ ]:
X_test_prepared = my_pipeline.transform(
    X_test
)

## 29. Final predictions

In [ ]:
final_predictions = forest_reg.predict(
    X_test_prepared
)

## 30. Final RMSE

In [ ]:
final_mse = mean_squared_error(
    y_test,
    final_predictions
)

final_rmse = np.sqrt(
    final_mse
)

print("Final test RMSE:", final_rmse)

## 31. Final R² score

In [ ]:
final_r2 = r2_score(
    y_test,
    final_predictions
)

print("Final test R²:", final_r2)

## 32. Compare actual and predicted values

In [ ]:
final_comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": final_predictions
})

final_comparison.head(20)

## 33. Prediction scatter plot

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    final_predictions,
    alpha=0.7
)

plt.xlabel("Actual MEDV")
plt.ylabel("Predicted MEDV")
plt.title("Actual vs Predicted House Prices")

plt.show()

## 34. Save the trained model

In [ ]:
joblib.dump(
    forest_reg,
    "Dragon.joblib"
)

In [ ]:
joblib.dump(
    my_pipeline,
    "DragonPipeline.joblib"
)

## 35. Load the model again

In [ ]:
loaded_model = joblib.load(
    "Dragon.joblib"
)

loaded_pipeline = joblib.load(
    "DragonPipeline.joblib"
)

## 36. Predict one property

In [ ]:
new_property = X_test.iloc[[0]]

In [ ]:
new_property

In [ ]:
new_property_prepared = loaded_pipeline.transform(
    new_property
)

In [ ]:
prediction = loaded_model.predict(
    new_property_prepared
)

prediction

In [ ]:
y_test.iloc[0]

# Better modern version: one complete pipeline

This approach combines preprocessing and modeling so that inference automatically applies the same transformations used during training.

## 37. Complete ML pipeline

In [ ]:
complete_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

In [ ]:
complete_pipeline.fit(
    housing_features,
    housing_labels
)

In [ ]:
predictions = complete_pipeline.predict(
    X_test
)

In [ ]:
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

print("RMSE:", rmse)

In [ ]:
joblib.dump(
    complete_pipeline,
    "dragon_real_estate_pipeline.joblib"
)

In [ ]:
model = joblib.load(
    "dragon_real_estate_pipeline.joblib"
)

In [ ]:
sample_house = X_test.iloc[[0]]

model.predict(sample_house)

## 38. Standalone model-usage example

In [ ]:
import joblib
import pandas as pd

In [ ]:
model = joblib.load(
    "dragon_real_estate_pipeline.joblib"
)

In [ ]:
house = pd.DataFrame([{
    "CRIM": 0.03,
    "ZN": 0.0,
    "INDUS": 7.0,
    "CHAS": 0,
    "NOX": 0.47,
    "RM": 6.5,
    "AGE": 70.0,
    "DIS": 5.0,
    "RAD": 4,
    "TAX": 300,
    "PTRATIO": 18.0,
    "B": 390.0,
    "LSTAT": 8.0
}])

In [ ]:
predicted_price = model.predict(
    house
)

print(
    "Predicted MEDV:",
    predicted_price[0]
)

In [ ]:
print(
    "Approximate predicted value: $",
    predicted_price[0] * 1000
)

## Notes

- Place `data.csv` in the same directory as this notebook before running it.
- Run the notebook from top to bottom.
- `MEDV` is the target variable.
- The final section includes a more modern single-pipeline approach that is safer for deployment.